In [1]:
import csv
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from PIL import Image
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import datasets

In [2]:
!pip install snntorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 4.0 MB/s eta 0:00:00


In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
from snntorch import utils
import matplotlib.pyplot as plt
import time
from sklearn.metrics import classification_report

In [4]:
from torchvision import datasets

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("/kaggle/input/fakeface-train-data-v2", transform=transform)
val_dataset   = datasets.ImageFolder("/kaggle/input/fakeface-valid-data-v2", transform=transform)
test_dataset = datasets.ImageFolder("/kaggle/input/fakeface-test-data-v2", transform=transform)

In [5]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader   = DataLoader(test_dataset, batch_size=16, shuffle=False,num_workers=2)

In [6]:
def rate_encode(images, num_steps):
    B, C, H, W = images.shape
    rand_vals = torch.rand((num_steps, B, C, H, W), device=images.device)
    spikes = (rand_vals < images.unsqueeze(0)).float()
    return spikes

In [7]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

beta = 0.95
spike_grad = surrogate.fast_sigmoid()
num_steps = 25  # số bước thời gian
batch_size = 128

class CSNN(nn.Module):
    def __init__(self):
        super(CSNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 12, 5)  # RGB
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=False)
        self.pool1 = nn.AvgPool2d(2)

        self.conv2 = nn.Conv2d(12, 32, 5)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=False)
        self.pool2 = nn.AvgPool2d(2)

        # Tính flatten_dim động
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 224, 224)
            dummy = self.pool1(self.conv1(dummy))
            dummy = self.pool2(self.conv2(dummy))
            flatten_dim = dummy.view(1, -1).size(1)

        self.fc1 = nn.Linear(flatten_dim, 10)  # ✅ chính xác theo shape

    def forward(self, x):
        batch_size = x.size(0)
        spike_train = rate_encode(x, num_steps)
        spk_out = None

        mem1 = torch.zeros(batch_size, 12, 220, 220, device=x.device)
        mem2 = torch.zeros(batch_size, 32, 106, 106, device=x.device)

        for step in range(num_steps):
            cur1 = self.conv1(spike_train[step])
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.pool1(spk1)
            cur2 = self.conv2(cur2)
            spk2, mem2 = self.lif2(cur2, mem2)

            cur3 = self.pool2(spk2)
            cur3 = cur3.view(cur3.size(0), -1)
            out = self.fc1(cur3)

            spk_out = out if spk_out is None else spk_out + out

        return spk_out / num_steps


In [8]:
from tqdm import tqdm
import time

def train(model, num_epochs):
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        total_loss = 0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for images, labels in train_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            train_bar.set_postfix(loss=loss.item())

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

    elapsed = time.time() - start_time
    print(f"Training time: {elapsed:.2f} seconds")

from sklearn.metrics import classification_report

def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\nAccuracy on test set: {100 * correct / total:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, digits=4))


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
csnn = CSNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(csnn.parameters(), lr=1e-3)

train(csnn, num_epochs=10)
evaluate(csnn)


Epoch 1/10: 100%|██████████| 3780/3780 [1:01:43<00:00,  1.02it/s, loss=0.332]


Epoch [1/10], Loss: 0.5430


Epoch 2/10: 100%|██████████| 3780/3780 [1:01:45<00:00,  1.02it/s, loss=0.307]


Epoch [2/10], Loss: 0.4044


Epoch 3/10: 100%|██████████| 3780/3780 [1:01:47<00:00,  1.02it/s, loss=0.65]


...

Epoch 9/10: 100%|██████████| 3780/3780 [1:01:51<00:00,  1.02it/s, loss=0.0644]


Epoch [9/10], Loss: 0.1755


Epoch 10/10: 100%|██████████| 3780/3780 [1:01:49<00:00,  1.02it/s, loss=0.23]

Epoch [10/10], Loss: 0.1595
Training time: 37083.90 seconds



Accuracy on test set: 82.22%

Classification Report:
              precision    recall  f1-score   support

           0     0.8385    0.7983    0.8179     20000
           1     0.8075    0.8462    0.8264     20000

    accuracy                         0.8223     40000
   macro avg     0.8230    0.8222    0.8221     40000
weighted avg     0.8230    0.8223    0.8221     40000



In [10]:
# Lưu toàn bộ mô hình (bao gồm cấu trúc và trọng số)
torch.save(csnn.state_dict(), "csnn_model.pth")
print("✅ CSNN model saved as 'csnn_model.pth'")


✅ CSNN model saved as 'csnn_model.pth'
